In [2]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy.stats import poisson
from sklearn.metrics import log_loss, brier_score_loss


In [3]:
df = pd.read_csv("../data/processed/features.csv")
df["date"] = pd.to_datetime(df["date"])

def season_from_date(d):
    return d.year + 1 if d.month >= 7 else d.year

df["season"] = df["date"].apply(season_from_date)

df = df[df["season"] <= 2024].copy()

train = df[df["season"] < 2024]
test = df[df["season"] == 2024]

train.shape, test.shape


((989, 11), (336, 11))

In [10]:
FEATURES = [
    "home_avg_scored",
    "home_avg_conceded",
    "away_avg_scored",
    "away_avg_conceded",
    "home_advantage",
]

X_test = sm.add_constant(test[FEATURES], has_constant="add")

X_test.head()


,const,home_avg_scored,home_avg_conceded,away_avg_scored,away_avg_conceded,home_advantage
989,1.0,1.4,1.6,0.4,1.4,1
990,1.0,1.0,1.6,0.4,1.6,1
991,1.0,0.6,1.0,0.4,1.2,1
992,1.0,0.4,1.4,0.6,1.4,1
993,1.0,0.4,1.8,0.6,1.4,1


In [11]:
home_model = sm.GLM(
    train["home_goals"],
    X_train,
    family=sm.families.Poisson()
).fit()

away_model = sm.GLM(
    train["away_goals"],
    X_train,
    family=sm.families.Poisson()
).fit()


In [12]:
test["lambda_home"] = home_model.predict(X_test)
test["lambda_away"] = away_model.predict(X_test)

test[["lambda_home", "lambda_away"]].head()


C:\Users\Shuay\AppData\Local\Temp\ipykernel_10776\2810846759.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test["lambda_home"] = home_model.predict(X_test)
C:\Users\Shuay\AppData\Local\Temp\ipykernel_10776\2810846759.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test["lambda_away"] = away_model.predict(X_test)


,lambda_home,lambda_away
989,1.460182,1.262211
990,1.426764,1.267274
991,1.431185,1.307503
992,1.392642,1.293515
993,1.372353,1.260741


In [13]:
def wdl_probs(lh, la, max_goals=10):
    hg = poisson.pmf(np.arange(max_goals + 1), lh)
    ag = poisson.pmf(np.arange(max_goals + 1), la)
    mat = np.outer(hg, ag)

    p_home = np.tril(mat, -1).sum()
    p_draw = np.trace(mat)
    p_away = np.triu(mat, 1).sum()

    total = p_home + p_draw + p_away
    return p_home / total, p_draw / total, p_away / total


test[["p_home", "p_draw", "p_away"]] = test.apply(
    lambda r: pd.Series(wdl_probs(r["lambda_home"], r["lambda_away"])),
    axis=1,
)

C:\Users\Shuay\AppData\Local\Temp\ipykernel_10776\2516131075.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test[["p_home", "p_draw", "p_away"]] = test.apply(
C:\Users\Shuay\AppData\Local\Temp\ipykernel_10776\2516131075.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test[["p_home", "p_draw", "p_away"]] = test.apply(
C:\Users\Shuay\AppData\Local\Temp\ipykernel_10776\2516131075.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_ind

In [14]:
(test[["p_home", "p_draw", "p_away"]].sum(axis=1).round(6) == 1).all()


np.True_

In [15]:
def outcome(row):
    if row.home_goals > row.away_goals:
        return 0
    if row.home_goals == row.away_goals:
        return 1
    return 2

y_true = test.apply(outcome, axis=1)
P = test[["p_home", "p_draw", "p_away"]]


In [16]:
from sklearn.metrics import log_loss, brier_score_loss

log_loss(y_true, P)


1.1017889756375827

In [17]:
test.to_csv("../data/processed/predictions.csv", index=False)
